# Validación cruzada (k-fold) — BETO + LoRA
Fija el learning_rate en el valor más estable visto en el grid search,
y explora combinaciones de `r` y `weight_decay` para elegir la mejor regularización.

In [ ]:
!git clone https://github.com/camistrika/BETO_HUMOR.git
%cd BETO_HUMOR
!pip install -e . -q

In [ ]:
!pip install -q torchao --upgrade
!pip install -q transformers peft datasets scikit-learn pyyaml

In [ ]:
%cd /content/BETO_HUMOR

In [ ]:
import numpy as np
import pandas as pd
from itertools import product
from sklearn.model_selection import StratifiedKFold
from transformers import AutoTokenizer
from google.colab import files

from betohumor.config import DataConfig, BetoConfig, LoraConfig
from betohumor.utils import set_seed
from betohumor.dataset import load_and_split
from betohumor.models.lora import build_beto_lora
from betohumor.hyperparam_search import run_one


## 1. Datos y configuraciones

In [ ]:
data_config = DataConfig()
beto_config = BetoConfig()
set_seed(data_config.seed)

df_train, df_val, df_test = load_and_split(data_config)

# Unimos train+val para repartir en folds. 
df_full = pd.concat([df_train, df_val]).reset_index(drop=True)

tokenizer = AutoTokenizer.from_pretrained(beto_config.base_model)
label_col = data_config.label_col

## 2A. Configuración de la búsqueda — weight_decay = 0.01


In [ ]:
LR_VALUES     = [1e-4, 3e-4, 1e-3]
R_VALUES      = [8, 16, 32]
WEIGHT_DECAYS = [0.01]
N_FOLDS       = 3

params_grid = [
    {'r': r, 'lora_alpha': r * 2, 'learning_rate': lr, 'weight_decay': wd}
    for lr, r, wd in product(LR_VALUES, R_VALUES, WEIGHT_DECAYS)
]

def builder(params):
    lora_config = LoraConfig(r=params['r'], lora_alpha=params['lora_alpha'])
    return build_beto_lora(beto_config, lora_config)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=data_config.seed)
params_grid

## 3. Correr la validación cruzada

In [ ]:
all_fold_results = []

for params in params_grid:
    fold_scores = []
    for fold_i, (train_idx, val_idx) in enumerate(skf.split(df_full, df_full[label_col])):
        run_name = f"r{params['r']}_lr{params['learning_rate']}_wd{params['weight_decay']}_fold{fold_i+1}"
        print(f"\n--- {run_name} ---")

        df_fold_train = df_full.iloc[train_idx].reset_index(drop=True)
        df_fold_val   = df_full.iloc[val_idx].reset_index(drop=True)

        output_dir = f"results/checkpoints/cv_lora/{run_name}"
        macro_f1, trainer = run_one(
            builder, params, beto_config, data_config,
            df_fold_train, df_fold_val, tokenizer, output_dir, seed=data_config.seed,
        )

        fold_scores.append(macro_f1)
        all_fold_results.append({**params, 'fold': fold_i + 1, 'macro_f1': macro_f1})
        print(f"Fold {fold_i+1} Macro F1: {macro_f1:.4f}")

    print(f"\n=== r={params['r']} lr={params['learning_rate']} wd={params['weight_decay']} -> Media: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f} ===")

## 4. Guardado bloque A (weight_decay = 0.01)

In [ ]:
df_folds = pd.DataFrame(all_fold_results)
df_folds.to_csv('results/cv_results_lora_wd01.csv', index=False)
df_folds

In [ ]:
files.download('results/cv_results_lora_wd01.csv')


## 2B. Configuración de la búsqueda - weight_decay = 0.05



In [ ]:
LR_VALUES     = [1e-4, 3e-4, 1e-3]
R_VALUES      = [8, 16, 32]
WEIGHT_DECAYS = [0.05]
N_FOLDS       = 3

params_grid = [
    {'r': r, 'lora_alpha': r * 2, 'learning_rate': lr, 'weight_decay': wd}
    for lr, r, wd in product(LR_VALUES, R_VALUES, WEIGHT_DECAYS)
]

def builder(params):
    lora_config = LoraConfig(r=params['r'], lora_alpha=params['lora_alpha'])
    return build_beto_lora(beto_config, lora_config)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=data_config.seed)
params_grid

## 5. Correr la validación cruzada (weight_decay = 0.05)

In [ ]:
all_fold_results = []

for params in params_grid:
    fold_scores = []
    for fold_i, (train_idx, val_idx) in enumerate(skf.split(df_full, df_full[label_col])):
        run_name = f"r{params['r']}_lr{params['learning_rate']}_wd{params['weight_decay']}_fold{fold_i+1}"
        print(f"\n--- {run_name} ---")

        df_fold_train = df_full.iloc[train_idx].reset_index(drop=True)
        df_fold_val   = df_full.iloc[val_idx].reset_index(drop=True)

        output_dir = f"results/checkpoints/cv_lora/{run_name}"
        macro_f1, trainer = run_one(
            builder, params, beto_config, data_config,
            df_fold_train, df_fold_val, tokenizer, output_dir, seed=data_config.seed,
        )

        fold_scores.append(macro_f1)
        all_fold_results.append({**params, 'fold': fold_i + 1, 'macro_f1': macro_f1})
        print(f"Fold {fold_i+1} Macro F1: {macro_f1:.4f}")

    print(f"\n=== r={params['r']} lr={params['learning_rate']} wd={params['weight_decay']} -> Media: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f} ===")

## 6. Guardado bloque B (weight_decay = 0.05)

In [ ]:
df_folds = pd.DataFrame(all_fold_results)
df_folds.to_csv('results/cv_results_lora_wd05.csv', index=False)
df_folds

In [ ]:
files.download('results/cv_results_lora_wd05.csv')


## 7. Unir ambos bloques


In [ ]:
df_folds_wd01 = pd.read_csv('results/cv_results_lora_wd01.csv')
df_folds_wd05 = pd.read_csv('results/cv_results_lora_wd05.csv')

df_folds = pd.concat([df_folds_wd01, df_folds_wd05]).reset_index(drop=True)
df_folds.to_csv('results/cv_results_lora.csv', index=False)

df_summary = (
    df_folds
    .groupby(['r', 'lora_alpha', 'learning_rate', 'weight_decay'])['macro_f1']
    .agg(mean_macro_f1='mean', std_macro_f1='std')
    .reset_index()
    .sort_values('mean_macro_f1', ascending=False)
)
df_summary.to_csv('results/cv_results_lora_summary.csv', index=False)
df_summary

In [ ]:
files.download('results/cv_results_lora.csv')
files.download('results/cv_results_lora_summary.csv')